# HU и модельно-условная доля воздуха в лёгком

**Статус:** контракт расчёта; исполняемый анализ реальных данных ещё не
реализован.

Этот этап должен получать статистики HU внутри зарегистрированного
ручного ROI лёгкого и оценивать долю воздуха при явно заданной модели.
Он не сегментирует лёгкие автоматически и не переводит результат в
электрическое удельное сопротивление. Сценарии электрических свойств
принадлежат только 20.04.


## Происхождение и исторический расчёт

В объединённой версии 20.01 до коммита fff4d3a использовалась пороговая
маска, предполагаемая нижняя треть правого лёгкого и формула

$$
f_{\mathrm{air}}=-\frac{HU}{1000}.
$$

Там же вычислялись электрические параметры по моделям
Максвелла–Гарнетта и Арчи и приводились численные выводы без сохранённых
outputs текущего кода. Эта цепочка не является активным результатом.
Пороговая маска остаётся историческим скетчем в
archive/legacy/20.90_Скетч_пороговой_КТ_сегментации.ipynb.

Связь «нижняя треть правого лёгкого = зона чувствительности боковой
сборки» не подтверждена регистрацией электродов. Региональный анализ
допустим только после 20.02 и с явно заданным анатомическим критерием.


## Модель HU → доля воздуха

Рабочая двухкомпонентная модель имеет вид

$$
f_{\mathrm{air}}=
\frac{HU_{\mathrm{tissue}}-HU}
     {HU_{\mathrm{tissue}}-HU_{\mathrm{air}}},
$$

где HU — значение в анализируемом вокселе или статистика ROI;
HU_air и HU_tissue — опорные значения воздуха и неаэрированной
тканевой компоненты в согласованных условиях реконструкции.

Это модельное допущение, а не прямое измерение объёмной доли воздуха.
Частный переход к HU_air = −1000 и HU_tissue = 0 пока не принят:
опорные значения, область применимости и источники должны быть проверены
по литературе из Zotero Heart_n_Lung и, если возможно, по калибровочным
областям конкретного тома. Значения вне интервала [0, 1] нельзя молча
обрезать; их доля должна выводиться как отдельный показатель нарушения
модели или качества входа.


## Статус литературного обоснования

Проверен локальный профиль Zotero Heart_Lung
(Zotero_Heart_n_Lung_Data), сначала подколлекция
«Изменение анатомии лёгких во время дыхания», затем полнотекстовый
индекс всей библиотеки.

- Chaudhary et al., *LungViT: Ensembling Cascade of Texture Sensitive
  Hierarchical Vision Transformers for Cross-Volume Chest CT
  Image-to-Image Translation* (Zotero BAET6GS9,
  DOI 10.1109/TMI.2024.3367321) используют КТ при TLC и RV,
  массосохраняющую регистрацию и заданные пороги HU для отдельных
  КТ-биомаркеров. Эта работа подтверждает необходимость фиксировать
  дыхательное состояние, регистрацию и протокол обработки, но не
  выводит непрерывную формулу HU → доля воздуха и не обосновывает
  выбранные опорные HU.
- Zhang, Tehrani, Wang, *A Biomechanical Modeling Guided CBCT
  Estimation Technique* (Zotero HWHNYCEB,
  DOI 10.1109/TMI.2016.2623745) применяют порог −250 HU только как
  начальный шаг сегментации с последующей ручной коррекцией и сохраняют
  HU для дозиметрической задачи. Эта работа также не обосновывает
  двухкомпонентное преобразование HU в долю воздуха.

Прямой первичный источник для линейной двухкомпонентной формулы и
выбора опорных значений в текущем локальном индексе не найден.
Следовательно, формула остаётся модельным допущением, а литературный
критерий допуска к реальному расчёту пока не выполнен.


## Обязательные входы

1. Принятый манифест 20.01 для однозначно выбранной серии, геометрии
   DICOM LPS и полного SHA-256 набора срезов.
2. Принятый манифест 20.02 с ручной маской Inobitec, преобразованием
   координат, хешами и неопределённостью регистрации.
3. Пиксельные DICOM-файлы именно того набора, который принят в 20.01.
4. Проверенное преобразование PixelData в HU через индивидуальные
   RescaleSlope и RescaleIntercept каждого среза.
5. Полный ручной ROI лёгкого или отдельно обоснованный
   анатомически зарегистрированный регион; автоматический порог не
   заменяет этот вход.
6. Состояние дыхания, статус контрастирования, ядро реконструкции и
   происхождение опорных HU.

До совпадения испытуемого, хешей серии и регистрационного манифеста
расчёт должен останавливаться.


## Предусмотренные вычисления и выход

Для принятого ROI необходимо получить:

- число вокселей и физический объём ROI;
- среднее, медиану, стандартное отклонение, межквартильный интервал и
  заранее заданные квантили HU;
- распределение и сводные статистики модельной доли воздуха;
- долю значений за пределами [0, 1] до каких-либо преобразований;
- чувствительность результата к HU_air и HU_tissue;
- хеши 20.01, 20.02, пиксельных данных, маски и конфигурации;
- версию кода, единицы, допущения, предупреждения и статус ручного QC.

Будущий канонический результат должен находиться вне Git по пути
derived_root/ct/hu_air_fraction/<subject_id>.json. params/ct.json и
derived_root/ct/legacy_hu_model/ct_properties.json относятся к
историческому смешанному интерфейсу и не являются выходами 20.03.


## Источники неопределённости

Необходимо учитывать раздельно:

- калибровку HU, реконструкционное ядро, контрастирование и состояние
  дыхания;
- частичный объём, шум изображения и неоднородность внутри ROI;
- ошибку ручной границы сегментации и преобразования координат;
- выбор полного или регионального ROI;
- опорные HU_air и HU_tissue и допустимость двухкомпонентной модели;
- отличие КТ-состояния от состояния электрической записи.

Неизвестные компоненты нельзя автоматически считать независимыми,
гауссовыми или сводить к одной дисперсии. Сначала требуется анализ
чувствительности к каждой составляющей и только затем обоснованная
совокупная оценка.


## Условия начала и завершения

Исполняемая реализация блокируется до выполнения всех условий:

1. существуют принятые реальные манифесты 20.01 и 20.02;
2. определён проверяемый формат зарегистрированного ROI и способ его
   наложения на DICOM-сетку;
3. в библиотеке Heart_n_Lung зафиксированы первичные источники для модели
   HU → доля воздуха и выбора опорных значений;
4. код проверен на синтетической геометрии и затем на разрешённых внешних
   данных без сохранения медицинских outputs в Git;
5. критерии принятия ROI, доли значений вне [0, 1] и чувствительности
   заданы до просмотра итоговых чисел.

Пока эти условия не выполнены, файл является контрактом будущего
расчёта. Исторические значения HU, доли воздуха и совпадение с
импедансной оценкой нельзя цитировать как валидированный результат.
